In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os 
import rasterio
from rasterio.transform import rowcol
from rasterio.transform import xy
from rasterio.windows import Window

# Data Explanation : 
1. Fauna : all animals distribution (amphibians , birds , mammals , reptiles) the data gives th nombre of birds , mammals ...etc in this area for example each pixel containe 1km²and the value is he number of animals in this area (form 0 --> 153  or 0 --> 178)

2. vegetation : the percentage of a specific land cover  (barren , evergreen, herbaceous) in speicific area (pixel)

3. human activities : gives the human modification in 1km² area between 0 and 1 

we can deduct somethings like : 
- High Species Richness + Low Human Modification → Biodiversity hotspots with low disturbance (ideal for conservation).
- High Species Richness + High Human Modification → Threatened ecosystems (urgent conservation priority).
- Low Species Richness + High Human Modification → Urban/agricultural areas with low ecological value.
- Low Species Richness + Low Human Modification → Natural areas with fewer species (e.g., deserts).

In [25]:
df = pd.read_csv("../results/Soil/final_Soil_dataset.csv")
df

,Unnamed: 0,X,Y,UD,LD,average profondeur,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,0,3.10700,36.684000,15.0,30.0,22.5,7.870000,0.170000,17.929000,46.172091,45.613249,8.214660,-0.005289,-0.749790,2.019000
1,1,3.10700,36.684000,30.0,60.0,45.0,7.774000,0.170000,17.207000,40.361496,37.281777,22.356727,-0.034471,-0.256559,1.503000
2,2,3.10700,36.684000,60.0,100.0,80.0,7.799000,0.170000,15.049000,49.608530,42.317043,8.074427,-0.069041,-0.788445,0.706000
3,3,3.13333,36.071667,15.0,30.0,22.5,7.950000,0.260000,40.610953,32.330400,40.460456,27.209144,0.097420,-0.074896,2.645000
4,4,3.13333,36.071667,30.0,60.0,45.0,8.096000,0.279000,40.610953,33.898341,39.248386,26.853273,0.063643,-0.101181,1.097000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2746,189,1.33000,36.220000,30.0,60.0,45.0,2.178122,4.704309,1.362000,2.818182,1.090909,2.363636,34.166000,17.992000,0.109091
2747,190,1.33000,36.220000,30.0,60.0,45.0,7.946000,3.350000,1.362000,23.351351,7.297297,15.324324,42.366000,23.120000,3.284000
2748,191,1.33000,36.220000,60.0,100.0,80.0,7.822000,3.350000,1.528000,8.648649,2.702703,5.675676,35.140000,23.120000,3.808000
2749,192,1.33000,36.220000,60.0,100.0,80.0,7.822000,3.350000,1.486000,30.000000,6.000000,13.333333,35.140000,23.120000,3.808000


In [26]:
df = df.drop(columns=["Unnamed: 0"])
df

,X,Y,UD,LD,average profondeur,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,3.10700,36.684000,15.0,30.0,22.5,7.870000,0.170000,17.929000,46.172091,45.613249,8.214660,-0.005289,-0.749790,2.019000
1,3.10700,36.684000,30.0,60.0,45.0,7.774000,0.170000,17.207000,40.361496,37.281777,22.356727,-0.034471,-0.256559,1.503000
2,3.10700,36.684000,60.0,100.0,80.0,7.799000,0.170000,15.049000,49.608530,42.317043,8.074427,-0.069041,-0.788445,0.706000
3,3.13333,36.071667,15.0,30.0,22.5,7.950000,0.260000,40.610953,32.330400,40.460456,27.209144,0.097420,-0.074896,2.645000
4,3.13333,36.071667,30.0,60.0,45.0,8.096000,0.279000,40.610953,33.898341,39.248386,26.853273,0.063643,-0.101181,1.097000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2746,1.33000,36.220000,30.0,60.0,45.0,2.178122,4.704309,1.362000,2.818182,1.090909,2.363636,34.166000,17.992000,0.109091
2747,1.33000,36.220000,30.0,60.0,45.0,7.946000,3.350000,1.362000,23.351351,7.297297,15.324324,42.366000,23.120000,3.284000
2748,1.33000,36.220000,60.0,100.0,80.0,7.822000,3.350000,1.528000,8.648649,2.702703,5.675676,35.140000,23.120000,3.808000
2749,1.33000,36.220000,60.0,100.0,80.0,7.822000,3.350000,1.486000,30.000000,6.000000,13.333333,35.140000,23.120000,3.808000


In [27]:
df_unique = df.copy()
df_unique = df_unique.drop_duplicates(subset=["X" ,"Y"])
df_unique 

,X,Y,UD,LD,average profondeur,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,3.107000,36.684000,15.0,30.0,22.5,7.870000,0.170000,17.929000,46.172091,45.613249,8.214660,-0.005289,-0.749790,2.019000
3,3.133330,36.071667,15.0,30.0,22.5,7.950000,0.260000,40.610953,32.330400,40.460456,27.209144,0.097420,-0.074896,2.645000
5,3.150000,36.724167,15.0,30.0,22.5,8.500000,2.632808,40.610953,13.000000,32.000000,55.000000,0.391206,0.626419,7.270000
7,2.880278,36.757500,5.0,15.0,10.0,8.590000,0.370000,40.610953,7.210000,11.570000,81.220000,0.205397,1.051727,4.100000
9,3.178361,36.704083,15.0,30.0,22.5,7.830000,2.632808,3.750000,27.239966,42.857252,29.902781,13.704000,0.674285,1.360000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2708,1.300000,36.210000,0.0,5.0,2.5,0.916667,0.053571,1.362000,3.690476,1.190476,2.023810,55.012000,17.992000,0.095238
2716,1.340000,36.200000,0.0,5.0,2.5,0.816327,0.051020,1.362000,3.469388,2.244898,0.612245,55.012000,17.992000,0.091837
2725,1.330000,36.210000,0.0,5.0,2.5,1.114286,0.100000,1.362000,4.142857,1.857143,3.000000,55.012000,17.992000,0.171429
2734,1.330000,36.200000,0.0,5.0,2.5,1.076872,4.726000,1.362000,4.428571,1.714286,3.714286,55.012000,17.992000,0.171429


In [44]:
def latlon_to_pixel(lon, lat, dataset ,transform):
    # Validate if the coordinate is within the image bounds
    if not (dataset.bounds.left <= lon <= dataset.bounds.right and
            dataset.bounds.bottom <= lat <= dataset.bounds.top):
            print(dataset.bounds.left , lon , dataset.bounds.right)
            print(dataset.bounds.bottom , lat , dataset.bounds.top)
    # Convert to pixel row & column
    row, col = rowcol(dataset.transform, lon, lat)
    return row, col

In [29]:
# windows Forming  for full one file (nefthou el file mera brk)
def window_forming(raster_path, df, height=7, width=7 , var=None):
    covariable_files = os.listdir(raster_path)
    covariable_num = len(covariable_files)

    datasets = [rasterio.open(os.path.join(raster_path, fname)) for fname in covariable_files]
    transforms = [ds.transform for ds in datasets]

    windows = []
    mid = height // 2

    for i in range(len(df)):
        print(f"Processing point {i}")
        lat = df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]
        window = np.empty((height, width, covariable_num))

        for idx, src in enumerate(datasets):
            transform = transforms[idx]
            if var :
                pixel_y, pixel_x = latlon_to_pixel(lon, lat, src, transform)
            else : 
                pixel_y, pixel_x = latlon_to_pixel(lat, lon, src, transform)

            try:
                win = Window(pixel_x - mid, pixel_y - mid, width, height)
                data = src.read(1, window=win)
                window[:, :, idx] = data

            except Exception as e:
                print(f"Error at point {i}, covariable {idx}")
                print(e)
                window[:, :, idx] = np.nan  # Fill with NaNs on failure

        windows.append(window)

    for ds in datasets:
        ds.close()

    return windows

In [30]:
# for devided tiff files 
def window_forming2(raster_path  , df ,height=7 , width=7 ):
    covariable_num = len(os.listdir(raster_path))
    windows = []

    for i in range(len(df)) :

        print(f"the dataset is in : {i}") 
        lat =  df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]

        window = np.empty((height, width, covariable_num))
        for index , j in enumerate(os.listdir(raster_path)):
            with rasterio.open(f"{raster_path}/{j}") as src :
                raster_data = src.read()
                transform = src.transform 
                
            try : 
                pixel_x ,pixel_y = latlon_to_pixel(lat , lon , src , transform)
                mid = height//2
                window[:,:,index] = raster_data[ 0 , pixel_x - mid : pixel_x + mid +1 , pixel_y - mid : pixel_y + mid +1 ]
                break 
            except Exception as e:
                print("can'found in ")
                continue 
        
        windows.append(window)
    return windows

In [31]:
def window_forming3(all_files  , df ,height=7 , width=7 ):
    covariable_num = len(all_files)
    windows = []
    for i in range(len(df)) :
        print(f"the dataset is in : {i}") 
        lat =  df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]

        window = np.empty((height, width, covariable_num))

        for index , i in enumerate(all_files):
            raster_data = i.raster_data
            pixel_x ,pixel_y = latlon_to_pixel(lat , lon ,  i.src , i.transform)
            print(pixel_x , pixel_y)
            mid = height//2
            try : 
                window[:,:,index] = raster_data[ pixel_x - mid : pixel_x + mid +1 , pixel_y - mid : pixel_y + mid +1 ]
            except Exception as e:
                print("error is here ")
                print(e)
        windows.append(window)
    return windows

### 1.Climat

In [32]:
climat_window = window_forming("../Resmapled/climat/" , df_unique)
climat_window = np.array(climat_window)
np.save("../Data_multi_CNN/climat.npy", climat_window)

Processing point 0
Processing point 1
Processing point 2
Processing point 3
Processing point 4
Processing point 5
Processing point 6
Processing point 7
Processing point 8
Processing point 9
Processing point 10
Processing point 11
Processing point 12
Processing point 13
Processing point 14
Processing point 15
Processing point 16
Processing point 17
Processing point 18
Processing point 19
Processing point 20
Processing point 21
Processing point 22
Processing point 23
Processing point 24
Processing point 25
Processing point 26
Processing point 27
Processing point 28
Processing point 29
Processing point 30
Processing point 31
Processing point 32
Processing point 33
Processing point 34
Processing point 35
Processing point 36
Processing point 37
Processing point 38
Processing point 39
Processing point 40
Processing point 41
Processing point 42
Processing point 43
Processing point 44
Processing point 45
Processing point 46
Processing point 47
Processing point 48
Processing point 49
Processing

In [33]:
climat_window.shape

(496, 7, 7, 6)

### 2. organisme

In [34]:
organisme = window_forming("../Resmapled/organisme/" , df_unique)
organisme = np.array(organisme)
print(organisme.shape)
np.save("../Data_multi_CNN/organisme.npy", organisme)

Processing point 0
Processing point 1
Processing point 2
Processing point 3
Processing point 4
Processing point 5
Processing point 6
Processing point 7
Processing point 8
Processing point 9
Processing point 10
Processing point 11
Processing point 12
Processing point 13
Processing point 14
Processing point 15
Processing point 16
Processing point 17
Processing point 18
Processing point 19
Processing point 20
Processing point 21
Processing point 22
Processing point 23
Processing point 24
Processing point 25
Processing point 26
Processing point 27
Processing point 28
Processing point 29
Processing point 30
Processing point 31
Processing point 32
Processing point 33
Processing point 34
Processing point 35
Processing point 36
Processing point 37
Processing point 38
Processing point 39
Processing point 40
Processing point 41
Processing point 42
Processing point 43
Processing point 44
Processing point 45
Processing point 46
Processing point 47
Processing point 48
Processing point 49
Processing

### 3. relief

In [35]:
relief = window_forming("../Resmapled/relief/" , df_unique)
relief = np.array(relief)
print(relief.shape)
np.save("../Data_multi_CNN/relief.npy", relief)

Processing point 0
Processing point 1
Processing point 2
Processing point 3
Processing point 4
Processing point 5
Processing point 6
Processing point 7
Processing point 8
Processing point 9
Processing point 10
Processing point 11
Processing point 12
Processing point 13
Processing point 14
Processing point 15
Processing point 16
Processing point 17
Processing point 18
Processing point 19
Processing point 20
Processing point 21
Processing point 22
Processing point 23
Processing point 24
Processing point 25
Processing point 26
Processing point 27
Processing point 28
Processing point 29
Processing point 30
Processing point 31
Processing point 32
Processing point 33
Processing point 34
Processing point 35
Processing point 36
Processing point 37
Processing point 38
Processing point 39
Processing point 40
Processing point 41
Processing point 42
Processing point 43
Processing point 44
Processing point 45
Processing point 46
Processing point 47
Processing point 48
Processing point 49
Processing

### 4. Spatial Position

In [ ]:
landuse_window = window_forming2("../Resmapled/saptialPosition/landuse/" , df_unique)
water_window = window_forming2("../Resmapled/saptialPosition/water/" , df_unique)
np.save("../Data_multi_CNN/landuse.npy", landuse_window)
np.save("../Data_multi_CNN/water.npy", water_window)

the dataset is in : 0
the dataset is in : 1
the dataset is in : 2
the dataset is in : 3
the dataset is in : 4
the dataset is in : 5
the dataset is in : 6
the dataset is in : 7
34.999583333333334 34.72694444 39.999583333333334
-0.0004166666666662877 6.273938889 4.999583333333334
the dataset is in : 8
34.999583333333334 34.71834444 39.999583333333334
-0.0004166666666662877 6.306361111000001 4.999583333333334
the dataset is in : 9
34.999583333333334 34.63508056 39.999583333333334
-0.0004166666666662877 5.101466667000004 4.999583333333334
the dataset is in : 10
34.999583333333334 34.63802222 39.999583333333334
-0.0004166666666662877 5.137791667000002 4.999583333333334
the dataset is in : 11
34.999583333333334 34.71680833 39.999583333333334
-0.0004166666666662877 5.369911111 4.999583333333334
the dataset is in : 12
34.999583333333334 34.72976111 39.999583333333334
-0.0004166666666662877 5.389869443999999 4.999583333333334
the dataset is in : 13
34.999583333333334 34.8095 39.999583333333334


In [40]:
np.array(landuse_window).shape

(496, 7, 7, 1)

In [41]:
np.array(water_window).shape

(496, 7, 7, 4)

### 4. Parent Material

In [45]:
parent_window = window_forming("../Resmapled/ParentMaterial/", df_unique)
np.save("../Data_multi_CNN/Parent_Material.npy", parent_window)
np.array(parent_window).shape

Processing point 0
Processing point 1
Processing point 2
Processing point 3
Processing point 4
Processing point 5
Processing point 6
Processing point 7
Processing point 8
Processing point 9
Processing point 10
Processing point 11
Processing point 12
Processing point 13
Processing point 14
Processing point 15
Processing point 16
Processing point 17
Processing point 18
Processing point 19
Processing point 20
Processing point 21
Processing point 22
Processing point 23
Processing point 24
Processing point 25
Processing point 26
Processing point 27
Processing point 28
Processing point 29
Processing point 30
Processing point 31
Processing point 32
Processing point 33
Processing point 34
Processing point 35
Processing point 36
Processing point 37
Processing point 38
Processing point 39
Processing point 40
Processing point 41
Processing point 42
Processing point 43
Processing point 44
Processing point 45
Processing point 46
Processing point 47
Processing point 48
Processing point 49
Processing

(496, 7, 7, 2)

In [ ]:
parent_window

In [ ]:
# shape file transformation
import geopandas as gpd
from rasterio.transform import from_bounds
from rasterio.features import rasterize

def transform_shape(shapefile_path , attribute , output_path):
    gdf = gpd.read_file(shapefile_path)
    gdf["category_numeric"], category_mapping = gdf[attribute].factorize()
    bounds = gdf.total_bounds  
    print("Bounds:", bounds)
    resolution = 0.008983  # -- 1KM
    width = int((bounds[2] - bounds[0]) / resolution)
    height = int((bounds[3] - bounds[1]) / resolution)

    print(f"Raster size: Width={width}, Height={height}")

    transform = from_bounds(*bounds, width, height)
    raster = rasterize(
        [(geom, value) for geom, value in zip(gdf.geometry, gdf["category_numeric"])],
        out_shape=(height, width),
        transform=transform,
        fill=-1, 
        dtype=rasterio.int32
    )

    tiff_path = output_path
    with rasterio.open(
        tiff_path, "w", 
        driver="GTiff",
        height=height, 
        width=width, 
        count=1,
        dtype=rasterio.uint8,
        crs=gdf.crs,  # Use same CRS as shapefile
        transform=transform
    ) as dst:
        dst.write(raster, 1)
    print(f"Saved 1 km raster to {tiff_path}")

In [ ]:
transform_shape("../Maps/ParentMaterial/Geomorphology/Algeria/Algeria_HG.shp" , "AlgGLG"  , "output_1km.tif")
geoMorpho_window = window_forming2("../Maps/ParentMaterial/Geomorphology/tiff" , df_unique)
np.save("../Data_multi_CNN/geoMorpho.npy", geoMorpho_window)
geoMorpho_window

In [78]:
class optimizerKhefoufi:
    def __init__(self , raster_data , transforme , src ):
        self.raster_data = raster_data
        self.transform = transforme
        self.src = src

def data_saver(path):
    files = []
    lat_min = 0 
    lat_max = 10

    lon_min = 30
    lon_max = 40


    for i in os.listdir(path):
        print(i)
        with rasterio.open(f"{path}/{i}") as src :
            raster_data = src.read()
            transform = src.transform 

            pixel_minx , pixel_miny = latlon_to_pixel(lat_min , lon_min ,src, transform)
            pixel_maxx , pixel_maxy = latlon_to_pixel(lat_max , lon_max , src , transform)
            print(pixel_minx , pixel_maxx , pixel_miny , pixel_maxy)
            
            raster_data = raster_data[0 , pixel_minx : pixel_maxx , pixel_miny : pixel_maxy]

            tiff_val = optimizerKhefoufi(raster_data , transform , src)
        files.append(tiff_val)
    return files
all_files = data_saver("../Maps/organisme/Microbial/SoilResp_HeterotrophicResp_1928/data")

FileNotFoundError: [WinError 3] The system cannot find the path specified: '../Maps/organisme/Microbial/SoilResp_HeterotrophicResp_1928/data'

In [ ]:
Microbial_window = window_forming3(all_files , df_unique)
np.save("../Data_multi_CNN/microbial.npy", Microbial_window)
Microbial_window

In [46]:
def pixel_to_latlon(row, col, dataset):
    lon, lat = xy(dataset.transform, row, col)
    return lat, lon

In [47]:
windows = []
raster_data_used = "../Resmapled/organisme/evergreen.tif"
with rasterio.open(raster_data_used) as src: 
    raster = src.read()
    transform = src.transform

    for index in range(len(df_unique)):
        lat = df_unique.iloc[index]["X"]
        lon = df_unique.iloc[index]["Y"]

        pixel_row, pixel_col = latlon_to_pixel(lat, lon, src, transform)
        window = np.zeros((7, 7, 2))  # Only initialize once

        for i in range(pixel_row - 3, pixel_row + 4):  # rows (Y)
            for j in range(pixel_col - 3, pixel_col + 4):  # cols (X)
                lat_val, lon_val = pixel_to_latlon(i, j, src)

                row_idx = i - (pixel_row - 3)
                col_idx = j - (pixel_col - 3)

                window[row_idx, col_idx, 0] = lat_val
                window[row_idx, col_idx, 1] = lon_val

        windows.append(window)

# Check result
print(windows[0].shape)
np.save("../Data_multi_CNN/latlon.npy", windows)

(7, 7, 2)


In [28]:
windows

[array([[[ 1.33324104, 36.18646279],
         [ 1.33324104, 36.19115959],
         [ 1.33324104, 36.19585638],
         [ 1.33324104, 36.20055318],
         [ 1.33324104, 36.20524997],
         [ 1.33324104, 36.20994677],
         [ 1.33324104, 36.21464357]],
 
        [[ 1.32874942, 36.18646279],
         [ 1.32874942, 36.19115959],
         [ 1.32874942, 36.19585638],
         [ 1.32874942, 36.20055318],
         [ 1.32874942, 36.20524997],
         [ 1.32874942, 36.20994677],
         [ 1.32874942, 36.21464357]],
 
        [[ 1.32425781, 36.18646279],
         [ 1.32425781, 36.19115959],
         [ 1.32425781, 36.19585638],
         [ 1.32425781, 36.20055318],
         [ 1.32425781, 36.20524997],
         [ 1.32425781, 36.20994677],
         [ 1.32425781, 36.21464357]],
 
        [[ 1.31976619, 36.18646279],
         [ 1.31976619, 36.19115959],
         [ 1.31976619, 36.19585638],
         [ 1.31976619, 36.20055318],
         [ 1.31976619, 36.20524997],
         [ 1.31976619, 36.209

In [ ]:
transform_shape("../Maps/ParentMaterial/Rock_type/geo7_2ag.shp" , "GLG"  , "../Maps/ParentMaterial/Rock_type/tiff/rocksType_1km.tif")
rocks_window = window_forming("../Maps/ParentMaterial/Rock_type/tiff", df_unique)
np.save("../Data_multi_CNN/rocks.npy", rocks_window)
rocks_window

,AREA,PERIMETER,GEO2_7G_,GEO2_7G_ID,GLG,geometry,category_numeric
0,2250.44404,966.65344,2,11939,SEA,"POLYGON ((5.40701 5.57146, 5.4109 5.4694, 5.42...",0
1,1.60529,7.58497,3,6,oth,"POLYGON ((15.64759 40.00266, 15.66 40, 15.676 ...",1
2,7.02790,33.05255,4,3,oth,"POLYGON ((19.89965 39.96591, 22.81549 39.8342,...",1
3,0.06592,1.66218,5,7,oth,"POLYGON ((19.67681 39.78778, 19.7106 39.7867, ...",1
4,0.90818,4.50544,6,4,oth,"POLYGON ((8.40944 39.76892, 9.63446 39.84441, ...",1
...,...,...,...,...,...,...,...
11972,0.01677,0.69987,11974,11935,SO,"POLYGON ((19.69988 -34.58595, 19.70456 -34.581...",31
11973,0.00456,0.35706,11975,11936,D,"POLYGON ((19.69988 -34.58595, 19.70059 -34.602...",27
11974,0.00239,0.20456,11976,11934,pCm,"POLYGON ((19.43258 -34.60229, 19.40437 -34.627...",11
11975,0.00641,0.35766,11977,11938,SO,"POLYGON ((19.84761 -34.67162, 19.85932 -34.661...",31


In [53]:
def outputs_building(df ,df_unique, atr):
    depth_ranges = [
        (0, 15),
        (5, 15),
        (15, 30),
        (30, 60),
        (60, 100),
        (100, 200)
    ]
    result_array = np.zeros((474, 6))
    depth_indices = {i: [] for i in range(len(depth_ranges))}

    for i, (lb, ub) in enumerate(depth_ranges):
        for j in range(lb, ub + 1):
            depth_indices[i].append(j)
    
    for _, row in df.iterrows():
        x, y, ud, ld, val = row['X'], row['Y'], row['UD'], row['LD'] , row[atr]
        for index, depth_range in depth_indices.items():
            if any(d in range(int(ud), int(ld) + 1) for d in depth_range):
                result_array[df_unique[(df_unique['X'] == x) & (df_unique['Y'] == y)].index[0], index] = val

    mean = df[atr].mean()
    result_array = np.nan_to_num(result_array, nan=mean)
    finals = np.array(result_array).reshape(474,6,1)
    for i in finals:
        print(i)
    
    print(finals.shape)
    return finals

In [56]:
df_unique = df_unique.reset_index()
df_unique

,index,X,Y,UD,LD,average profondeur,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,0,1.32,36.20,0,5,2.5,1.316667,0.233333,8.544,7.333333,2.166667,2.500000,20.256,4.843655,0.216667
1,8,1.28,38.00,0,5,2.5,1.316667,0.233333,12.040,7.333333,2.166667,2.500000,1.940,2.457421,0.216667
2,16,1.31,36.20,0,5,2.5,0.914634,0.060976,8.544,5.365854,1.829268,1.707317,20.256,4.843655,0.146341
3,25,1.30,36.19,0,5,2.5,1.054054,0.054054,8.544,6.621622,2.702703,1.216216,20.256,4.843655,0.162162
4,34,1.32,36.22,0,5,2.5,1.218750,0.093750,8.544,4.218750,1.406250,2.500000,20.256,4.843655,0.234375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
469,2612,2.41,27.18,0,23,11.5,8.430000,20.000000,6.720,0.460000,6.800000,87.752000,59.032,8.702000,0.800000
470,2615,2.25,27.35,0,20,10.0,7.630000,1.150000,4.182,1.822000,5.584000,88.490000,42.300,8.896000,2.380000
471,2619,2.41,27.96,0,10,5.0,6.720000,3.710000,4.182,1.500000,2.484000,91.626000,47.416,8.374000,1.060000
472,2624,2.45,27.20,0,3,1.5,6.800000,3.750000,4.182,1.500000,2.484000,91.626000,39.720,8.374000,1.320000


In [58]:
df_unique = df_unique.drop(columns=["index"])
df_unique

,X,Y,UD,LD,average profondeur,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,1.32,36.20,0,5,2.5,1.316667,0.233333,8.544,7.333333,2.166667,2.500000,20.256,4.843655,0.216667
1,1.28,38.00,0,5,2.5,1.316667,0.233333,12.040,7.333333,2.166667,2.500000,1.940,2.457421,0.216667
2,1.31,36.20,0,5,2.5,0.914634,0.060976,8.544,5.365854,1.829268,1.707317,20.256,4.843655,0.146341
3,1.30,36.19,0,5,2.5,1.054054,0.054054,8.544,6.621622,2.702703,1.216216,20.256,4.843655,0.162162
4,1.32,36.22,0,5,2.5,1.218750,0.093750,8.544,4.218750,1.406250,2.500000,20.256,4.843655,0.234375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
469,2.41,27.18,0,23,11.5,8.430000,20.000000,6.720,0.460000,6.800000,87.752000,59.032,8.702000,0.800000
470,2.25,27.35,0,20,10.0,7.630000,1.150000,4.182,1.822000,5.584000,88.490000,42.300,8.896000,2.380000
471,2.41,27.96,0,10,5.0,6.720000,3.710000,4.182,1.500000,2.484000,91.626000,47.416,8.374000,1.060000
472,2.45,27.20,0,3,1.5,6.800000,3.750000,4.182,1.500000,2.484000,91.626000,39.720,8.374000,1.320000


In [ ]:
df

,X,Y,UD,LD,average profondeur,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,1.32,36.20,0.0,5.0,2.5,1.316667,0.233333,8.544000,7.333333,2.166667,2.500,20.256000,4.843655,0.216667
1,1.32,36.20,5.0,15.0,10.0,2.633333,0.466667,8.126000,14.666667,4.333333,5.000,20.256000,4.853527,0.433333
2,1.32,36.20,15.0,30.0,22.5,3.950000,0.700000,16.186191,22.000000,6.500000,7.500,19.814000,0.138530,0.650000
3,1.32,36.20,30.0,60.0,45.0,5.333333,0.333333,11.876000,30.666667,10.666667,10.000,34.226429,-0.162834,0.800000
4,1.32,36.20,60.0,100.0,80.0,2.666667,0.166667,10.206000,15.333333,5.333333,5.000,74.816000,0.248837,0.400000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627,2.43,27.19,0.0,8.0,4.0,7.390000,2.860000,4.182000,1.500000,5.584000,88.490,41.090000,8.896000,1.430000
2628,2.43,27.19,8.0,34.0,21.0,7.650000,4.870000,4.352000,0.406000,14.524000,85.736,46.690000,8.896000,0.530000
2629,2.43,27.19,34.0,43.0,38.5,7.990000,5.480000,3.246000,0.214000,9.618000,87.536,38.250000,7.354000,1.060000
2630,2.43,27.19,43.0,60.0,51.5,8.040000,4.070000,2.914000,0.194000,4.398000,92.376,34.214000,7.354000,0.000000


In [61]:
for i in df.columns[5:]:
    print(i)
    finals = outputs_building(df ,df_unique, i)
    np.save(f'../Data_multi_CNN/output{i}.npy', np.array(finals))

pH
[[0.23235294]
 [0.23235294]
 [6.97058824]
 [0.77878676]
 [8.06      ]
 [8.06      ]]
[[3.95      ]
 [3.95      ]
 [5.33333333]
 [4.44444444]
 [8.        ]
 [8.        ]]
[[2.74390244]
 [2.74390244]
 [4.43333333]
 [5.05555556]
 [8.        ]
 [8.        ]]
[[3.77419355]
 [3.77419355]
 [7.159375  ]
 [0.740625  ]
 [0.740625  ]
 [0.740625  ]]
[[3.65625]
 [3.65625]
 [6.825  ]
 [7.8    ]
 [7.8    ]
 [7.8    ]]
[[3.12857143]
 [3.12857143]
 [4.5625    ]
 [2.80769231]
 [4.49230769]
 [4.49230769]]
[[3.        ]
 [3.        ]
 [4.82352941]
 [6.04324324]
 [8.6       ]
 [8.6       ]]
[[3.95      ]
 [3.95      ]
 [5.92682927]
 [5.94871795]
 [2.05128205]
 [2.05128205]]
[[3.075     ]
 [3.075     ]
 [2.96666667]
 [5.93333333]
 [9.        ]
 [9.        ]]
[[3.375     ]
 [3.375     ]
 [0.92718615]
 [5.18862745]
 [5.18862745]
 [5.18862745]]
[[3.85      ]
 [3.85      ]
 [4.74      ]
 [2.63333333]
 [8.06      ]
 [8.06      ]]
[[3.22058824]
 [3.22058824]
 [5.2       ]
 [2.56925742]
 [4.53012821]
 [4.530128